# Fase 2: RAG

## 1. Chatbot básico

### 1.1. Configuración del cliente de Groq

In [2]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
API_KEY = os.getenv("GROQ_KEY")

client = OpenAI(base_url="https://api.groq.com/openai/v1", api_key=API_KEY)

### 1.2. Comparar variantes de system prompt

In [3]:
SYSTEM_PROMPTS = {
    "Asistente General": (
        "Eres un asistente útil que responde preguntas de forma clara y comprensible. "
        "Proporciona respuestas correctas y breves. "
        "No es necesario usar un lenguaje técnico."
    ),
    "Asistente Especializado": (
        "Eres un asistente experto en privacidad digital, protección de datos y términos de servicio. "
        "Responde de forma breve, precisa, clara y estructurada. "
        "Utiliza lenguaje técnico cuando sea necesario, pero mantén la claridad. "
    ),
    "Asistente Restrictivo": (
        "Eres un asistente especializado en privacidad, protección de datos y términos de servicio."
        "Responde únicamente a preguntas relacionadas con estos temas."
        "Si la pregunta no está relacionada con privacidad, protección de datos o términos digitales, debes responder exactamente:"
        "\"No dispongo de información suficiente para responder con precisión.\""
        "No respondas preguntas fuera de este dominio bajo ninguna circunstancia."
        "Usa un tono formal y respuestas breves y estructuradas."
    ),
}

PREGUNTAS = [
    # Preguntas del dominio
    "¿Qué son las cookies y para qué se utilizan?",
    "¿Qué derechos tiene un usuario según el RGPD?",
    "¿Qué datos personales recopila Spotify y con qué finalidad?",

    # Preguntas fuera del dominio (para analizar alucinaciones y límites del modelo)
    "¿Cómo funciona la fotosíntesis en las plantas?",
    "¿Cómo se relacionan la teoría de la relatividad y la mecánica cuántica en la descripción del espacio-tiempo?"
]

In [6]:
MODEL = "llama-3.1-8b-instant"

def generar_respuesta(pregunta, system_prompt, temperature=1.0):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": pregunta}
        ],
        max_completion_tokens=300,
        temperature=temperature
    )
    return response.choices[0].message.content.strip()

In [37]:
# Comparar respuestas de los tres asistentes para cada pregunta
for pregunta in PREGUNTAS:
    print(f"\nPregunta: {pregunta}\n{'='*50}")
    for asistente, system_prompt in SYSTEM_PROMPTS.items():
        respuesta = generar_respuesta(pregunta, system_prompt)
        print(f"{asistente}:\n{respuesta}\n{'-'*50}")


Pregunta: ¿Cómo se relacionan la teoría de la relatividad y la mecánica cuántica en la descripción del espacio-tiempo?
Asistente General:
La teoría de la relatividad y la mecánica cuántica son dos marcos teóricos fundamentales de la física que se relacionan en la descripción del espacio-tiempo de manera importante.

La teoría de la relatividad general, desarrollada por Albert Einstein, describe el espacio-tiempo como una sola entidad curva y dinámica, influenciada por la presencia de masa y energía. Esto implica que el espacio y el tiempo no son absolutos, sino que están interconectados y se curvan en presencia de objetos masivos.

Por otro lado, la mecánica cuántica describe los comportamientos de los objetos a nivel atómico y subatómico, donde las partículas se comportan de manera probabilística y no determinista. La mecánica cuántica también tiene un impacto en la descripción del espacio-tiempo, ya que sugiere que el espacio-t
--------------------------------------------------
Asis

### 1.3. Comparar parámetros de generación

In [11]:
# Se selecciona una pregunta que permita evaluar precisión + control + alucinaciones
pregunta = PREGUNTAS[2]
system_prompt = SYSTEM_PROMPTS["Asistente Restrictivo"]

print("Temperatura = 1.0 (respuestas más creativas y variadas)\n")
print(generar_respuesta(pregunta, system_prompt, temperature=1.0))

print(f"\n{'='*50}\nTemperatura = 0.6 (respuestas más coherentes y menos creativas)\n")
print(generar_respuesta(pregunta, system_prompt, temperature=0.6))

print(f"\n{'='*50}\nTemperatura = 0.2 (respuestas más precisas y menos creativas)\n")
print(generar_respuesta(pregunta, system_prompt, temperature=0.2))

Temperatura = 1.0 (respuestas más creativas y variadas)

Spotify recopila diversos tipos de datos personales, incluyendo:

1. **Información de cuenta**:nombre de usuario, contraseña, correo electrónico, fecha de nacimiento y género.
2. **Datos de actividad**: reproducciones de canciones, listas de reproducción, búsqueda de contenidos y duración de la sesión.
3. **Información de dispositivo**: tipo de dispositivo, idioma, versión de sistema operativo y datos de localización.
4. **Datos de pago**: información de tarjeta de crédito o débito, método de pago y saldo.

El fin principal de recopilar estos datos es proporcionar servicios a los usuarios, como:

- Personalizar recomendaciones de música.
- Permitir el acceso a contenido premium.
- Mejorar la experiencia de usuario y el rendimiento del servicio.
- Cumplir con obligaciones legales y regulaciones, como las relacionadas con la protección de menores.

No obstante, Spotify ha publicado una política de privacidad detallada, disponible e

 **Conclusión:** La mejor combinación para RAG es "Asistente Restrictivo" como system prompt + temperature=0.2

## 2. Chunking + embeddings

En este apartado se compararán los métodos de chunking por tamaño fijo y recursivo mostrando como quedan los chunks y evaluando cualitativamente la calidad del retrieval. 

Para ello, se usará solo uno de los documentos del corpus. 

Una vez seleccionada la estrategia de chunking más óptima, se implementará en el apartado 3 el pipeline completo almacenando los embeddings en una base de datos vectorial.

### 2.1. Comparar configuraciones de chunking 

In [45]:
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter

# Chunking por tamaño fijo
splitter_fijo = CharacterTextSplitter(
    separator="\n",
    chunk_size=500,      # máximo de caracteres por chunk
    chunk_overlap=50,    # solapamiento entre chunks consecutivos
    length_function=len
)

# Chunking recursivo
splitter_recursivo = RecursiveCharacterTextSplitter(
    chunk_size=500,      # máximo de caracteres por chunk
    chunk_overlap=100,   # 20% de solapamiento
    # Prueba separadores de mayor a menor granularidad
    separators=["\n# ", "\n## ", "\n### ", "\n---\n", "\n\n", "\n", ". ", " "],
    length_function=len,
)

In [52]:
# Comparar chunks generados por cada método
def mostrar_chunks(chunks):
    print(f"Chunks generados: {len(chunks)}\n")
    # Mostrar chunks del 20 al 25 para ver variedad
    for i, chunk in enumerate(chunks[19:25]):
        print(f"--- Chunk {i + 20} ({len(chunk)} chars) ---")
        print(chunk)
        print()

# Para comparar ambos metodos, se usa uno de los documentos del corpus
with open("corpus/guides/gdpr_explained.md", "r", encoding="utf-8") as f:
    texto = f.read()

# Generar chunks usando ambos métodos
chunks_fijo = splitter_fijo.split_text(texto)
chunks_recursivo = splitter_recursivo.split_text(texto)


Created a chunk of size 760, which is longer than the specified 500


In [53]:
print("Chunks generados por el método de tamaño fijo:")
mostrar_chunks(chunks_fijo)

Chunks generados por el método de tamaño fijo:
Chunks generados: 34

--- Chunk 20 (331 chars) ---
### Data protection by design and by default
From now on, everything you do in your organization must, “by design and by default,” consider data protection. Practically speaking, this means you must consider the data protection principles in the design of any new product or activity. The GDPR covers this principle in Article 25 .

--- Chunk 21 (297 chars) ---
Suppose, for example, you’re launching a new app for your company. You have to think about what personal data the app could possibly collect from users, then consider ways to minimize the amount of data and how you will secure it with the latest technology.
### When you’re allowed to process data

--- Chunk 22 (424 chars) ---
### When you’re allowed to process data
Article 6 lists the instances in which it’s legal to process person data. Don’t even think about touching somebody’s personal data — don’t collect it, don’t store it, don’t

In [54]:
print("Chunks generados por el método recursivo:")
mostrar_chunks(chunks_recursivo)

Chunks generados por el método recursivo:
Chunks generados: 43

--- Chunk 20 (377 chars) ---
---

[IMAGE: 7_GPDR_principles.webp]

Descripción:
The image provides an overview of the seven GDPR data protection principles.  

- Purpose Limitations  
- Lawfulness  
- Data Minimization  
- Storage Limitations  
- Accountability  
- Accuracy  
- Confidentiality  

This infographic illustrates the fundamental aspects of data protection required under GDPR compliance.

---

--- Chunk 21 (287 chars) ---
### Accountability

The GDPR says data controllers have to be able to demonstrate they are GDPR compliant. And this isn’t something you can do after the fact: If you think you are compliant with the GDPR but can’t show how, then you’re not GDPR compliant. Among the ways you can do this:

--- Chunk 22 (466 chars) ---
- Designate data protection responsibilities to your team.
- Maintain detailed documentation of the data you’re collecting, how it’s used, where it’s stored, which employee is respo

In [47]:
from sentence_transformers import SentenceTransformer

modelo_emb = SentenceTransformer("intfloat/multilingual-e5-base")

# Generar embeddings de chunks 
def generar_embeddings(chunks):
    list_chunks = ["passage: " + c for c in chunks]
    embeddings = modelo_emb.encode(
        list_chunks,
        normalize_embeddings=True
    )
    return embeddings

embeddings_chunks_fijo = generar_embeddings(chunks_fijo)
embeddings_chunks_recursivo = generar_embeddings(chunks_recursivo)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6620.50it/s]
XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [55]:
# Evaluar cualitativamente la calidad del retrieval para cada método de chunking 
import numpy as np

QUERIES = [
    "What is the GDPR?",
    "Which are the data protection principles?",
    "What are the rights of a user according to the GDPR?",
]

def mostrar_similitudes(embeddings, chunks, queries, top_k=3):
    """Muestra los chunks más similares a cada query según los embeddings."""
    for query in queries:
        query_emb = modelo_emb.encode(["query: " + query], normalize_embeddings=True)[0]

        # Similitud coseno entre la query y todos los documentos
        similitudes = modelo_emb.similarity(query_emb, embeddings)[0].numpy()
        
        # Índices ordenados de mayor a menor similitud
        top_indices = np.argsort(similitudes)[-top_k:][::-1]

        print(f"\nQuery: {query}\n{'-'*50}")
        for i, idx in enumerate(top_indices):
            print(f"{i + 1}. (sim={similitudes[idx]:.3f}) {chunks[idx]}")


In [56]:
print("Similitudes para chunks generados por el método de tamaño fijo:")
mostrar_similitudes(embeddings_chunks_fijo, chunks_fijo, QUERIES)

Similitudes para chunks generados por el método de tamaño fijo:

Query: What is the GDPR?
--------------------------------------------------
1. (sim=0.907) The General Data Protection Regulation (GDPR) is the toughest privacy and security law in the world. Though it was drafted and passed by the European Union (EU), it imposes obligations onto organizations anywhere, so long as they target or collect data related to people in the EU. The regulation was put into effect on May 25, 2018. The GDPR will levy harsh fines against those who violate its privacy and security standards, with penalties reaching into the tens of millions of euros.
2. (sim=0.890) If you’ve found this page — “what is the GDPR?” — chances are you’re looking for a crash course. Maybe you haven’t even found the document itself yet (tip: here’s the full regulation ). Maybe you don’t have time to read the whole thing. This page is for you. In this article, we try to demystify the GDPR and, we hope, make it less overwhelmi

In [57]:
print("Similitudes para chunks generados por el método recursivo:")
mostrar_similitudes(embeddings_chunks_recursivo, chunks_recursivo, QUERIES)

Similitudes para chunks generados por el método recursivo:

Query: What is the GDPR?
--------------------------------------------------
1. (sim=0.907) The General Data Protection Regulation (GDPR) is the toughest privacy and security law in the world. Though it was drafted and passed by the European Union (EU), it imposes obligations onto organizations anywhere, so long as they target or collect data related to people in the EU. The regulation was put into effect on May 25, 2018. The GDPR will levy harsh fines against those who violate its privacy and security standards, with penalties reaching into the tens of millions of euros.
2. (sim=0.893) # What is GDPR, the EU’s new data protection law?

What is the GDPR? Europe’s new data privacy and security law includes hundreds of pages’ worth of new requirements for organizations around the world. This GDPR overview will help you understand the law and determine what parts of it apply to you.
3. (sim=0.881) With the GDPR, Europe is signalin

**Conclusiones:** El método recursivo genera chunks más coherentes y relevantes para las queries, lo que mejora la calidad del retrieval. El método de tamaño fijo produce chunks arbitrarios que pueden cortar información importante, resultando en similitudes más bajas y peor retrieval al devolver chunks menos relevantes.

Por ello, en el pipeline completo se aplicará chunking recursivo con 20% de solapamiento entre chunks.